# CounterFeint - Official HuggingFace GRPO Training

GRPO fine-tuning of a small Qwen3 model as the **Investigator** in the CounterFeint multi-agent fraud-detection environment.

## What this notebook does

1. Loads `Qwen/Qwen3-0.6B` (smallest Qwen3) in 4-bit + LoRA on a T4 GPU
2. Collects multi-agent rollouts in-process (no HTTP server, no Ollama):
   - **Fraudster**: `ReactiveFraudster` (scripted, deterministic)
   - **Investigator**: our trainable `HFInvestigator` wrapping Qwen3-0.6B
   - **Auditor**: `HeuristicAuditor` (scripted, rule-based)
3. Trains the Investigator with TRL's `GRPOTrainer` using a verifiable proxy reward
4. Evaluates before/after on held-out seeds
5. Pushes the LoRA adapter to the Hugging Face Hub

## Why this design

- **Scripted Fraudster** (no LLM opponent for first training proof) keeps rollouts cheap and deterministic. Following the recommendation in `COUNTERFEINT_REVIEW.md` - `LLM_FRAUDSTER_RATIO = 0.0` for the first run.
- **In-process env** (`RefereeEnvironment`) skips the websocket/uvicorn server. Same env code as the production HTTP path; just no transport.
- **Two-phase pipeline** (collect rollouts -> GRPO on dataset) is more sample-efficient than online `rollout_func` for our 20-step episodes.

## Run modes

| MODE  | episodes | epochs | est. time on T4 |
|-------|----------|--------|-----------------|
| `smoke` | 6 (2 seeds x 3 tasks) | 1 | ~5 min (just verify pipeline) |
| `demo`  | 30 (10 seeds x 3 tasks) | 1 | ~30 min |
| `full`  | 100 | 2 | ~2-3 hours |

Set `MODE` in the config cell below.

## Section 1 - Setup and Install

Installs CounterFeint (editable) plus the training dependencies (`trl`, `peft`, `bitsandbytes`, `accelerate`, `datasets`, `matplotlib`, `nest_asyncio`).

If you're on **Colab** the first cell does a fresh clone + install. On **HuggingFace Spaces** with a notebook environment the same command works because the repo is already in the workspace.

After install, restart the runtime if Colab asks - then re-run from Section 2.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")

# Locate the CounterFeint repo root. On Colab we clone it; otherwise we
# expect the notebook to be opened from inside the existing repo.
if IN_COLAB:
    repo_dir = Path("/content/OpenEnv-Hackathon")
    if not repo_dir.exists():
        # Replace this URL with your fork/clone if you've pushed somewhere else.
        subprocess.run(
            ["git", "clone", "https://github.com/Abhijithreddydasari/OpenEnv-Hackathon.git", str(repo_dir)],
            check=True,
        )
    REPO_ROOT = repo_dir
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "counterfeint" / "server").exists()),
        here,
    )

print(f"REPO_ROOT = {REPO_ROOT}")
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
os.chdir("/home/user/app/counterfeint")
def pip_install(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

print("Installing CounterFeint (editable) ...")
pip_install(["-e", "."])

print("Installing training dependencies ...")
pip_install(["-r", "counterfeint/requirements-train.txt"])

print("Done. If Colab asks you to restart the runtime, restart it then resume from Section 2.")

In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except Exception as exc:
    print(f"[hf-login] skipped: {exc}")
    print("You can also export HF_TOKEN as an env var before launching the notebook.")

In [ ]:
import torch
import nest_asyncio

nest_asyncio.apply()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. GRPO will be extremely slow on CPU.")

## Section 2 - Configuration

All knobs in one place. Default is `MODE = "proper"` for a real training run. Use `MODE = "smoke"` first if you want to verify the pipeline cheaply.

| MODE   | episodes | epochs | est. T4 wall-time | use case |
|--------|----------|--------|-------------------|----------|
| smoke  | 6   | 1 | ~5 min      | pipeline check, no real learning |
| demo   | 30  | 1 | ~30-45 min  | first weak signal, plot sanity |
| proper | 60  | 2 | ~2-3 hrs    | first real grader_score lift |
| full   | 60  | 3 | ~6-8 hrs    | aim for the 0.8+ ceiling |

You can also override from the shell: `COUNTERFEINT_MODE=demo` then run the cell.

**Baseline reference** (untrained Qwen2.5-1.5B against scripted Fraudster, from the latest `replay_*.md` logs):

| Task   | grader_score |
|--------|--------------|
| task_1 | ~0.84-0.85 |
| task_2 | ~0.64 |
| task_3 | ~0.32 (link_accounts is the weak spot) |

GRPO's job: lift task_3 toward 0.55+ and hold task_1/2 above 0.8.

In [ ]:
import json
import os
from typing import Dict, List

# On HF Spaces, ensure CWD is the counterfeint package root so
# relative output paths like 'outputs/' land inside the repo tree.
for _candidate in ['/data/counterfeint', '/home/user/app/counterfeint']:
    if os.path.isdir(_candidate):
        os.chdir(_candidate)
        break
print(f'Working directory: {os.getcwd()}')

# Pick MODE here. With a 12-hr budget on a T4 ($0.60/hr -> ~$7), one
# "proper" run is the right default. "smoke" is for verifying the
# pipeline in 5 min before committing to the real run.
MODE = os.environ.get("COUNTERFEINT_MODE", "proper")

BASE_MODEL = "Qwen/Qwen3.5-0.8B"
TRAINED_TAG = "counterfeint-investigator-qwen35-08b-grpo"

# Hub repo where the LoRA adapter will be pushed at the end. Replace
# `<your-username>` with your HF username before running with push_to_hub=True.
HUB_REPO_ID = f"<your-username>/{TRAINED_TAG}"

OUTPUT_DIR = Path("outputs") / TRAINED_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Episode / dataset knobs ----------------------------------------
TASKS = ["task_1", "task_2", "task_3"]

# Each MODE picks: seeds-per-task, training epochs, eval seed count.
MODE_PRESETS = {
    "smoke":  {"seeds": [11, 13],            "epochs": 1, "eval_seeds": [101]},
    "demo":   {"seeds": list(range(11, 21)), "epochs": 1, "eval_seeds": [101, 103, 107]},
    "proper": {"seeds": list(range(11, 21)) * 2,
               "epochs": 2, "eval_seeds": [101, 103, 107, 109, 113]},
    "full":   {"seeds": list(range(11, 31)),
               "epochs": 3, "eval_seeds": [101, 103, 107, 109, 113]},
}
preset = MODE_PRESETS[MODE]
train_seeds = preset["seeds"]
EVAL_SEEDS  = preset["eval_seeds"]

TRAIN_SEEDS_BY_TASK: Dict[str, List[int]] = {t: list(train_seeds) for t in TASKS}
EVAL_SEEDS_BY_TASK:  Dict[str, List[int]] = {t: list(EVAL_SEEDS) for t in TASKS}

# ---- LoRA / model knobs ---------------------------------------------
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 128       # actions are tiny JSON; 128 is plenty
TEMPERATURE = 0.3          # low temperature => low fallback rate

# r=16 gives 4x the capacity vs r=8 at <1% extra params on a 0.6B base;
# task_3 (link_accounts) needs the extra plasticity.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ---- GRPO knobs ------------------------------------------------------
LEARNING_RATE = 2e-5
NUM_GENERATIONS = 4         # group size for GRPO
KL_BETA = 0.01
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8
MAX_COMPLETION_LEN = 256
MAX_PROMPT_LEN = 1024
NUM_EPOCHS = preset["epochs"]
SAVE_STEPS = 50
LOG_STEPS = 1
SEED = 7

# Skip the BEFORE eval (saves ~5-10 min by not reloading the base model
# a second time). Flip to True only on your first run, or when you change
# the base model.
RUN_BEFORE_EVAL = True

PUSH_TO_HUB = False         # flip to True after setting HUB_REPO_ID

config_summary = {
    "mode": MODE,
    "base_model": BASE_MODEL,
    "n_episodes": len(TASKS) * len(train_seeds),
    "n_eval_episodes": len(TASKS) * len(EVAL_SEEDS),
    "epochs": NUM_EPOCHS,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "targets": LORA_TARGETS},
    "grpo": {
        "lr": LEARNING_RATE, "num_generations": NUM_GENERATIONS,
        "kl_beta": KL_BETA, "max_completion_len": MAX_COMPLETION_LEN,
    },
    "output_dir": str(OUTPUT_DIR),
    "push_to_hub": PUSH_TO_HUB,
    "run_before_eval": RUN_BEFORE_EVAL,
}
print(json.dumps(config_summary, indent=2))

## Section 3 - Load Qwen3-0.6B + LoRA

We load the model in 4-bit (NF4) to fit on a T4 with room for activations and the LoRA optimizer state, then attach a small LoRA on the attention projections only.

`HFInvestigator` wraps the base model + tokenizer with the same prompt-builder, JSON parser, and scripted-fallback wiring that the production HTTP investigator uses, so the trained policy ports cleanly.

`enable_thinking=False` tells Qwen3's chat template to skip its chain-of-thought wrapper - we want the model to output the JSON action directly.

In [ ]:
from counterfeint.agents import HFInvestigator

print(f"Loading {BASE_MODEL} (load_in_4bit={LOAD_IN_4BIT}) ...")
hf_investigator = HFInvestigator.from_pretrained(
    BASE_MODEL,
    load_in_4bit=LOAD_IN_4BIT,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    do_sample=True,
    enable_thinking=False,
)
print("Loaded.")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if LOAD_IN_4BIT:
    hf_investigator.model = prepare_model_for_kbit_training(hf_investigator.model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)
hf_investigator.model = get_peft_model(hf_investigator.model, lora_cfg)
hf_investigator.model.print_trainable_parameters()

In [ ]:
probe_msgs = [
    {"role": "system", "content": "You output one line of valid JSON only."},
    {"role": "user",   "content": "Reply with {\"ok\": true}"},
]
probe_text = hf_investigator._call_chat(probe_msgs)
print(f"Smoke-test completion (first 200 chars):\n{probe_text[:200]!r}")

## Section 4 - Collect rollouts

Run multi-agent episodes in-process (`RefereeEnvironment` directly) with the trainable Investigator + scripted Fraudster + heuristic Auditor. We record every Investigator `act()` call as a `(prompt, completion, reward)` row.

**Reward shaping** (in `records_to_samples`): the episode-level Investigator reward is split across turns 80/20 between verdict-class and investigate-class actions. This gives stronger gradient on the actions that actually move the grader.

The smoke run with 6 episodes typically takes ~3-5 min on a T4.

In [ ]:
import time
from counterfeint.training import (
    collect_dataset_in_process,
    samples_to_hf_dataset,
)
from counterfeint.scripted import HeuristicAuditor, ReactiveFraudster

t0 = time.perf_counter()
samples = collect_dataset_in_process(
    hf_investigator=hf_investigator,
    seeds_by_task=TRAIN_SEEDS_BY_TASK,
    fraudster_factory=lambda: ReactiveFraudster(seed=42),
    auditor_factory=lambda: HeuristicAuditor(),
    max_steps=80,
    show_trace=False,
)
elapsed = time.perf_counter() - t0
print(f"\nCollected {len(samples)} training rows in {elapsed:.1f}s "
      f"({elapsed / max(1, len(samples)):.2f}s/row).")
print(f"Investigator fallback: {hf_investigator.fallback_count}/{hf_investigator.call_count}")

In [ ]:
if not samples:
    raise RuntimeError(
        "No usable Investigator turns were collected. Every step must have "
        "fallen back to the scripted policy. Check fallback_count above and "
        "the smoke-test completion in Section 3."
    )

import statistics

rewards = [s.reward for s in samples]
grader_scores = sorted({(s.task_id, s.seed, s.terminal_grader_score) for s in samples})
class_counts = {"verdict": 0, "investigate": 0}
for s in samples:
    class_counts[s.metadata.get("action_class", "investigate")] += 1

print(f"Rows:                 {len(samples)}")
print(f"Reward mean:          {statistics.mean(rewards):+.4f}")
print(f"Reward stdev:         {statistics.pstdev(rewards):+.4f}")
print(f"Reward min/max:       {min(rewards):+.4f} / {max(rewards):+.4f}")
print(f"Action classes:       {class_counts}")
print(f"\nPer-episode grader_score:")
for task_id, seed, gs in grader_scores:
    print(f"  {task_id} seed={seed:>3}: {gs:+.4f}")

In [ ]:
# Drop any rows where the recorded completion was empty / not-JSON. The
# proxy reward already penalizes invalid JSON, but training on a dataset
# of mostly-broken completions wastes batches; the model learns more from
# good examples. We keep at most 90% of original rows.
def _looks_like_json(completion):
    if not completion:
        return False
    stripped = completion.strip()
    return stripped.startswith("{") and "action_type" in stripped

clean_samples = [s for s in samples if _looks_like_json(s.completion)]
dropped = len(samples) - len(clean_samples)
if dropped:
    print(f"Filtered out {dropped}/{len(samples)} rows with invalid completions.")
samples = clean_samples or samples  # fall back if filter would empty everything

from counterfeint.agents.prompts import INVESTIGATOR_SYSTEM_PROMPT
train_dataset = samples_to_hf_dataset(samples, system_prompt=INVESTIGATOR_SYSTEM_PROMPT)
print(train_dataset)
print("\nFirst row preview:")
preview = train_dataset[0]
print(f"  prompt    (first 300 chars): {preview['prompt'][:300]!r}")
print(f"  completion (first 200 chars): {preview['completion'][:200]!r}")
print(f"  reward                      : {preview['reward']:+.4f}")
print(f"  task_id / seed              : {preview['task_id']} / {preview['seed']}")
print(f"  action_class                : {preview['metadata'].get('action_class')}")

## Section 5 - GRPO Training

The `proxy_reward_fn` scores any `(prompt, completion)` pair without touching the env, so it works on TRL's freshly-generated rollouts. It rewards:

- **+0.6** for valid `AdReviewAction` JSON (schema validity)
- **+0.1** each for `ad_id` / `linked_ad_id` referenced in the prompt (coherence)
- **+0.2** for matching the recorded gold action class (verdict / investigate / link)
- **up to +0.6** for matching the recorded gold decision, scaled by the episode's `grader_score` quality

Range: roughly `[-0.5, +2.0]`.

`GRPOTrainer` runs `num_generations=4` rollouts per prompt, scores them with this reward, and updates LoRA via group-relative policy optimization with `KL_BETA=0.01` regularization to the frozen base.

In [ ]:
from counterfeint.training import build_gold_lookup, make_proxy_reward_fn

gold_lookup = build_gold_lookup(samples)
proxy_reward_fn = make_proxy_reward_fn(gold_lookup=gold_lookup)
print(f"Built gold lookup with {len(gold_lookup)} entries.")

# Quick sanity check: score a known-good completion.
sample_prompt = samples[0].prompt
sample_completion = samples[0].completion
sample_reward = proxy_reward_fn([sample_prompt], [sample_completion])
print(f"Proxy reward on a recorded completion: {sample_reward}")

In [ ]:
import inspect
from trl import GRPOConfig, GRPOTrainer

# Build the config kwargs incrementally so we tolerate older/newer TRL
# versions: ``max_prompt_length`` was removed in TRL >=1.0 (the prompt is
# now truncated by tokenizer.model_max_length).
_grpo_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "trl_checkpoints"),
    learning_rate=LEARNING_RATE,
    num_generations=NUM_GENERATIONS,
    beta=KL_BETA,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_completion_length=MAX_COMPLETION_LEN,
    num_train_epochs=NUM_EPOCHS,
    save_steps=SAVE_STEPS,
    logging_steps=LOG_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)
_grpo_params = set(inspect.signature(GRPOConfig.__init__).parameters)
if "max_prompt_length" in _grpo_params:
    _grpo_kwargs["max_prompt_length"] = MAX_PROMPT_LEN
else:
    hf_investigator.tokenizer.model_max_length = max(
        MAX_PROMPT_LEN + MAX_COMPLETION_LEN,
        getattr(hf_investigator.tokenizer, "model_max_length", 0) or 0,
    )

if "temperature" in _grpo_params:
    _grpo_kwargs["temperature"] = 0.9

trl_config = GRPOConfig(**_grpo_kwargs)

trainer = GRPOTrainer(
    model=hf_investigator.model,
    args=trl_config,
    train_dataset=train_dataset,
    reward_funcs=[proxy_reward_fn],
    processing_class=hf_investigator.tokenizer,
)
if hasattr(trainer, "generation_config"):
    trainer.generation_config.temperature = 0.9
    trainer.generation_config.do_sample = True
print("GRPOTrainer ready (generation temperature=0.9).")

In [ ]:
train_t0 = time.perf_counter()
train_result = trainer.train()
print(f"\nTraining took {(time.perf_counter() - train_t0) / 60:.1f} min.")
print(train_result)

In [ ]:
lora_dir = OUTPUT_DIR / "lora_adapter"
lora_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(lora_dir))
hf_investigator.tokenizer.save_pretrained(str(lora_dir))

with open(OUTPUT_DIR / "training_config.json", "w") as f:
    json.dump(config_summary, f, indent=2)

print(f"Saved LoRA adapter + tokenizer to: {lora_dir}")

## Section 6 - Loss / reward curves

The submission requirement asks for "loss and reward plots from a real run." We pull TRL's `log_history` (which contains per-`logging_steps` records of `loss`, `reward`, `kl`, `learning_rate`, ...) and render them with matplotlib.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
log_path = OUTPUT_DIR / "log_history.json"
with open(log_path, "w") as f:
    json.dump(log_history, f, indent=2, default=str)
print(f"Wrote {log_path} ({len(log_history)} entries).")

steps_loss, losses = [], []
steps_rew,  rewards_log = [], []
steps_kl,   kls = [], []

for entry in log_history:
    step = entry.get("step")
    if step is None:
        continue
    if "loss" in entry:
        steps_loss.append(step); losses.append(entry["loss"])
    if "reward" in entry:
        steps_rew.append(step); rewards_log.append(entry["reward"])
    if "kl" in entry:
        steps_kl.append(step); kls.append(entry["kl"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps_loss, losses, color="tab:red")
axes[0].set_title("GRPO loss")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps_rew, rewards_log, color="tab:green")
axes[1].set_title("Mean reward")
axes[1].set_xlabel("step"); axes[1].set_ylabel("reward")
axes[1].grid(True, alpha=0.3)

axes[2].plot(steps_kl, kls, color="tab:blue")
axes[2].set_title("KL divergence")
axes[2].set_xlabel("step"); axes[2].set_ylabel("KL")
axes[2].grid(True, alpha=0.3)

fig.suptitle(f"GRPO training - {BASE_MODEL} - mode={MODE}")
fig.tight_layout()

plot_path = OUTPUT_DIR / "training_curves.png"
fig.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved plot to {plot_path}")

## Section 7 - Before/After evaluation

Run the **trained** Investigator on held-out seeds and compare the grader_score against:

- **Before**: the *base* Qwen3-0.6B without LoRA (i.e. what we started with)

We use the in-process driver again (no HTTP server needed). The Fraudster + Auditor are deterministic so the only thing that changes between Before and After is the Investigator's policy.

In [ ]:
from counterfeint.server.referee import RefereeEnvironment


def run_eval_episode(investigator, *, task_id: str, seed: int, max_steps: int = 80):
    """In-process episode runner for evaluation. Returns flat metrics dict."""
    env = RefereeEnvironment()
    env.reset_match(task_id=task_id, seed=seed)

    fraudster = ReactiveFraudster(seed=42)
    auditor = HeuristicAuditor()

    role_handlers = {
        "fraudster_turn":   (fraudster,    env.build_fraudster_observation,    env.step_as_fraudster),
        "investigator_turn":(investigator, env.build_investigator_observation, env.step_as_investigator),
        "audit_phase":      (auditor,      env.build_auditor_observation,      env.step_as_auditor),
    }

    step_idx = 0
    n_verdicts = 0
    while env.phase in role_handlers:
        current_phase = env.phase
        policy, build_obs, step_fn = role_handlers[current_phase]
        if current_phase != "audit_phase" and step_idx >= max_steps:
            break
        obs = build_obs().model_dump()
        for slot in ("last_prompt", "last_completion", "last_error"):
            if hasattr(policy, slot):
                setattr(policy, slot, None)
        try:
            action = policy.act(obs)
        except Exception:
            break
        try:
            step_fn(action)
        except Exception:
            break
        if current_phase != "audit_phase":
            step_idx += 1
        if current_phase == "investigator_turn" and getattr(action, "action_type", None) in ("verdict", "link_accounts"):
            n_verdicts += 1

    state = env.state
    return {
        "task_id": task_id,
        "seed": seed,
        "grader_score": getattr(state, "grader_score", None),
        "end_reason":   getattr(state, "end_reason", None),
        "stopped_phase": env.phase,
        "n_verdicts": n_verdicts,
        "investigator_fallback": getattr(investigator, "fallback_count", 0),
    }


def run_eval_suite(investigator, *, label: str, seeds_by_task):
    print(f"\n=== Evaluating {label} ===")
    results = []
    for task_id, seeds in seeds_by_task.items():
        for seed in seeds:
            fb_before = getattr(investigator, "fallback_count", 0)
            r = run_eval_episode(investigator, task_id=task_id, seed=seed)
            r["investigator_fallback_delta"] = (
                getattr(investigator, "fallback_count", 0) - fb_before
            )
            results.append(r)
            print(f"  {task_id} seed={seed:>3}: grader={r['grader_score']}  "
                  f"verdicts={r['n_verdicts']}  fallback+={r['investigator_fallback_delta']}")
    return results

In [ ]:
after_results = run_eval_suite(
    hf_investigator,
    label="AFTER (trained Qwen3-0.6B + LoRA)",
    seeds_by_task=EVAL_SEEDS_BY_TASK,
)

In [ ]:
import gc

if not RUN_BEFORE_EVAL:
    print("RUN_BEFORE_EVAL=False - skipping the base-model reload pass.")
    print("Using a synthetic BEFORE = 0.0 baseline (or paste numbers from a")
    print("previous run into `before_results` below if you have them).")
    before_results = [
        {
            "task_id": r["task_id"], "seed": r["seed"], "grader_score": 0.0,
            "end_reason": "skipped", "stopped_phase": "skipped",
            "n_verdicts": 0, "investigator_fallback": 0,
            "investigator_fallback_delta": 0,
        }
        for r in after_results
    ]
else:
    del trainer
    del hf_investigator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Trained model unloaded; reloading the *base* Qwen3-0.6B for the BEFORE eval ...")

    baseline_investigator = HFInvestigator.from_pretrained(
        BASE_MODEL,
        load_in_4bit=LOAD_IN_4BIT,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True,
        enable_thinking=False,
    )

    before_results = run_eval_suite(
        baseline_investigator,
        label="BEFORE (base Qwen3-0.6B, no LoRA)",
        seeds_by_task=EVAL_SEEDS_BY_TASK,
    )

In [ ]:
def _scores(rows):
    return [r["grader_score"] or 0.0 for r in rows]

before_scores = _scores(before_results)
after_scores  = _scores(after_results)

eval_summary = {
    "n_eval_episodes": len(before_results),
    "before": {
        "mean_grader_score": sum(before_scores) / max(1, len(before_scores)),
        "verdicts_total":   sum(r["n_verdicts"] for r in before_results),
        "fallback_total":   sum(r["investigator_fallback_delta"] for r in before_results),
    },
    "after": {
        "mean_grader_score": sum(after_scores) / max(1, len(after_scores)),
        "verdicts_total":   sum(r["n_verdicts"] for r in after_results),
        "fallback_total":   sum(r["investigator_fallback_delta"] for r in after_results),
    },
}
eval_summary["delta_grader_score"] = (
    eval_summary["after"]["mean_grader_score"] - eval_summary["before"]["mean_grader_score"]
)

print(json.dumps(eval_summary, indent=2))

with open(OUTPUT_DIR / "eval_summary.json", "w") as f:
    json.dump({
        "summary": eval_summary,
        "before": before_results,
        "after":  after_results,
    }, f, indent=2)

In [ ]:
labels = [f"{r['task_id']}/{r['seed']}" for r in before_results]
x = list(range(len(labels)))
width = 0.4

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.6), 5))
ax.bar([xi - width/2 for xi in x], before_scores, width, label="Before (base)",     color="tab:gray")
ax.bar([xi + width/2 for xi in x], after_scores,  width, label="After (trained)",  color="tab:green")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("grader_score")
ax.set_title(f"Before vs After - {BASE_MODEL} - mode={MODE}")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()

eval_plot_path = OUTPUT_DIR / "eval_plot.png"
fig.savefig(eval_plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {eval_plot_path}")

## Section 8 - Push to Hub (optional)

Set `PUSH_TO_HUB = True` in Section 2 and replace `<your-username>` in `HUB_REPO_ID` to push the LoRA adapter + tokenizer + training/eval artefacts to a public HF Hub repo. The README on your CounterFeint env Space can then link to this adapter.

In [ ]:
if not PUSH_TO_HUB:
    print("PUSH_TO_HUB=False — skipping. Flip it in Section 2 once you're ready.")
else:
    if HUB_REPO_ID.startswith("<"):
        raise RuntimeError(
            "Set HUB_REPO_ID in Section 2 to '<your-hf-username>/<repo-name>' "
            "before pushing."
        )

    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(repo_id=HUB_REPO_ID, private=False, exist_ok=True)
    print(f"Created/found repo {HUB_REPO_ID}")

    api.upload_folder(
        repo_id=HUB_REPO_ID,
        folder_path=str(OUTPUT_DIR),
        commit_message=f"GRPO {MODE} run on {BASE_MODEL}",
    )
    print(f"Uploaded {OUTPUT_DIR} -> {HUB_REPO_ID}")

## Done

You should now have, under `OUTPUT_DIR`:

- `lora_adapter/` - the trained PEFT adapter (`adapter_config.json` + safetensors) and tokenizer
- `training_curves.png` - GRPO loss / reward / KL curves
- `eval_plot.png` - before/after grader_score per held-out episode
- `eval_summary.json` - mean grader_score before/after, fallback counts
- `log_history.json` - raw TRL training log
- `training_config.json` - the config that produced this run

To run a bigger experiment: change `MODE = "demo"` in Section 2, restart the kernel, and re-run all cells. With the $30 HF Spaces budget you have ~20 demo-mode runs to iterate on hyperparameters / model size.